# Badge 4: Data Lake Workshop: Lesson 4: Working with Non-Loaded Unstructured Data

Directory tables, joins, CATALOG and CATALOG_FOR_WEBSITE, and the optional Streamlit app.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, swapped for `BADGE4_DLKW_ADMIN` (ACCOUNTADMIN), `BADGE4_DLKW_ROLE` (SYSADMIN) and `BADGE4_DLKW_WH` (COMPUTE_WH).
- **SQL equivalent** replaces a UI-only step; **From screenshot** marks code the course shows only as an image; **Derived solution** marks challenge answers the course does not give.
- DORA checks are not run here; see `dora/dora_tests.ipynb`.
- Files the course has learners download are on `@UTIL_DB.PUBLIC.BADGE4_DOWNLOADS`, standing in for the learner's Downloads folder.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE4_DLKW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE4_DLKW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE4_DLKW_WH"      # stands in for the course's COMPUTE_WH


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result

In [ ]:
%%sql -r session_context
use role {{SYSADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
use schema ZENAS_ATHLEISURE_DB.PRODUCTS;
select current_role(), current_warehouse(), current_database(), current_schema();

## Page: 📓 Non-Loaded Data is Easy!

> **Course:** 🎯 Run a LIST command on the SWEATSUITS stage you created.

In [ ]:
%%sql -r list_sweatsuits
list @sweatsuits;

## Page: 📓 Unstructured Non-Loaded Data

> ⚠️ **Audit note:** The course says this query gives an error message (DLKW_038). It runs as given and reports what happens.

In [ ]:
%%sql -r select_dollar_1_png
execute immediate 'begin
  execute immediate ''select $1
from @sweatsuits/purple_sweatsuit.png'';
  return ''ran without error'';
exception when other then
  return ''Error (as the course predicts): '' || sqlerrm;
end;';

## Page: 🥋 Querying Non-Loaded Unstructured Data

In [ ]:
%%sql -r metadata_columns
select metadata$filename, metadata$file_row_number
from @sweatsuits/purple_sweatsuit.png;

> **Course:** 🎯 Write a Query That Returns Something More Like a List Command: GROUP BY the file name with MAX or COUNT (DLKW_040).

In [ ]:
%%sql -r list_like_query
-- Derived solution (columns from DLKW_040)
select metadata$filename, max(metadata$file_row_number) as number_of_rows
from @sweatsuits
group by metadata$filename;

## Page: 📓 File Formats? Nope, Directory Tables!

In [ ]:
%%sql -r directory_table
select * 
from directory(@sweatsuits);

## Page: 📓 Do Functions Work on Directory Tables?

In [ ]:
%%sql -r directory_functions
select REPLACE(relative_path, '_', ' ') as no_underscores_filename
, REPLACE(no_underscores_filename, '.png') as just_words_filename
, INITCAP(just_words_filename) as product_name
from directory(@sweatsuits);

> **Course:** 🎯 Nest 3 Functions into 1 Statement: a single column named PRODUCT_NAME (DLKW_049, 050).

In [ ]:
%%sql -r nested_functions
-- From screenshot DLKW_054 (the complete nested expression)
select INITCAP(REPLACE(REPLACE(relative_path,'_',' '), '.png')) as product_name
from directory(@sweatsuits);

## Page: 🥋 Can you Join Directory Tables with Other Tables?

In [ ]:
%%sql -r create_sweatsuits_table
--create an internal table for some sweatsuit info
create or replace table zenas_athleisure_db.products.sweatsuits (
	color_or_style varchar(25),
	file_name varchar(50),
	price number(5,2)
);

--fill the new table with some data
insert into  zenas_athleisure_db.products.sweatsuits 
          (color_or_style, file_name, price)
values
 ('Burgundy', 'burgundy_sweatsuit.png',65)
,('Charcoal Grey', 'charcoal_grey_sweatsuit.png',65)
,('Forest Green', 'forest_green_sweatsuit.png',64)
,('Navy Blue', 'navy_blue_sweatsuit.png',65)
,('Orange', 'orange_sweatsuit.png',65)
,('Pink', 'pink_sweatsuit.png',63)
,('Purple', 'purple_sweatsuit.png',64)
,('Red', 'red_sweatsuit.png',68)
,('Royal Blue',	'royal_blue_sweatsuit.png',65)
,('Yellow', 'yellow_sweatsuit.png',67);

> **Course:** 🎯 Can You Join These? (DLKW_053, 054)

In [ ]:
%%sql -r join_directory_table
-- From screenshot DLKW_054, blur filled in
select INITCAP(REPLACE(REPLACE(relative_path,'_',' '), '.png')) as product_name
, *
from directory(@sweatsuits) d
join sweatsuits s
on d.relative_path = s.file_name;

## Page: 🥋 3 Way Join - CROSS JOIN Included!

> **Course:** 🎯 Replace the * With a List of Columns: create a view named PRODUCT_LIST (DLKW_055: PRODUCT_NAME, FILE_NAME, COLOR_OR_STYLE, PRICE, FILE_URL).

In [ ]:
%%sql -r create_product_list
-- Derived solution (columns from DLKW_055)
create view product_list as
select INITCAP(REPLACE(REPLACE(relative_path,'_',' '), '.png')) as product_name
, s.file_name
, s.color_or_style
, s.price
, d.file_url
from directory(@sweatsuits) d
join sweatsuits s
on d.relative_path = s.file_name;

In [ ]:
%%sql -r cross_join
select * 
from product_list p
cross join sweatsuit_sizes;

> **Course:** 🎯 Convert Your Select Statement to a View: CATALOG, in Zena's database and PRODUCTS schema, owned by SYSADMIN; it should return 180 rows.

In [ ]:
%%sql -r create_catalog
-- Derived solution
create view catalog as
select * 
from product_list p
cross join sweatsuit_sizes;

In [ ]:
%%sql -r catalog_rows
select count(*) as catalog_rows from catalog;

## Page: 🤖 DORA DLKW03

### 🤖 DORA check DLKW03 goes here
The course has the learner run **DLKW03** (CATALOG view returns 180 rows) at this point.
It is in `dora/dora_tests.ipynb`, section **DLKW03**, as a local check plus the exact DORA code.

## Page: 🥋 Do WHAT You Can, WHERE You Can, HOW You Can

In [ ]:
%%sql -r create_upsell_mapping
-- Add a table to map the sweatsuits to the sweat band sets
create table zenas_athleisure_db.products.upsell_mapping
(
sweatsuit_color_or_style varchar(25)
,upsell_product_code varchar(10)
);

--populate the upsell table
insert into zenas_athleisure_db.products.upsell_mapping
(
sweatsuit_color_or_style
,upsell_product_code 
)
VALUES
('Charcoal Grey','SWT_GRY')
,('Forest Green','SWT_FGN')
,('Orange','SWT_ORG')
,('Pink', 'SWT_PNK')
,('Red','SWT_RED')
,('Yellow', 'SWT_YLW');

## Page: 🥋 Kludges Galore? Or Relentless Resourcefulness?

In [ ]:
%%sql -r create_catalog_for_website
-- Zena needs a single view she can query for her website prototype
create view catalog_for_website as 
select color_or_style
,price
,file_name
, get_presigned_url(@sweatsuits, file_name, 3600) as file_url
,size_list
,coalesce('Consider: ' ||  headband_description || ' & ' || wristband_description, 'Consider: White, Black or Grey Sweat Accessories')  as upsell_product_desc
from
(   select color_or_style, price, file_name
    ,listagg(sizes_available, ' | ') within group (order by sizes_available) as size_list
    from catalog
    group by color_or_style, price, file_name
) c
left join upsell_mapping u
on u.sweatsuit_color_or_style = c.color_or_style
left join sweatband_coordination sc
on sc.product_code = u.upsell_product_code
left join sweatband_product_line spl
on spl.product_code = sc.product_code;

## Page: 🤖 DORA DLKW04

### 🤖 DORA check DLKW04 goes here
The course has the learner run **DLKW04** (CATALOG_FOR_WEBSITE upsell descriptions) at this point.
It is in `dora/dora_tests.ipynb`, section **DLKW04**, as a local check plus the exact DORA code.

## Page: 🖼️ Zena's Ready to Meet Up with Mel!

Story only.

## Page: 🥋 Build Zena's SiS Prototype App (OPTIONAL)

> **Course:** UI steps (DLKW_062, 063): Projects → Streamlit → **+ Streamlit App**; App title **Zena's Amazing Athleisure Catalog**; app location **ZENAS_ATHLEISURE_DB / PRODUCTS**; warehouse **COMPUTE_WH**; Create. Delete ALL the sample code (DLKW_063) and paste the course's app code.

In [ ]:
%%sql -r create_streamlit_app
-- SQL equivalent of the Create Streamlit App dialog (DLKW_062)
use role {{SYSADMIN_ROLE}};
create streamlit ZENAS_ATHLEISURE_DB.PRODUCTS."Zena's Amazing Athleisure Catalog"
  runtime_name = 'SYSTEM$WAREHOUSE_RUNTIME'
  query_warehouse = {{WAREHOUSE}}
  title = 'Zena''s Amazing Athleisure Catalog';
alter streamlit ZENAS_ATHLEISURE_DB.PRODUCTS."Zena's Amazing Athleisure Catalog" add live version from last;

In [ ]:
# SQL equivalent of pasting the app code into the Streamlit editor: written into the app's live version
import io
from snowflake.snowpark.context import get_active_session
session = get_active_session()
APP_CODE = 'import streamlit as st\nfrom snowflake.snowpark.context import get_active_session\nfrom snowflake.snowpark.functions import col\nimport pandas as pd\n\nst.title("Zena\'s Amazing Athleisure Catalog")\n\nsession = get_active_session()\n\n# get a list of colors for a drop list selection\ntable_colors = session.sql("select color_or_style from catalog_for_website")\npd_colors = table_colors.to_pandas()\n\n# Oyt the list of colors into a drop list selector \noption = st.selectbox(\'Pick a sweatsuit color or style:\', pd_colors)\n\n# We\'ll build the image caption now, since we can\nproduct_caption = \'Our warm, comfortable, \' + option + \' sweatsuit!\'\n\n# use the color selected to go back and get all the info from the database\ntable_prod_data = session.sql("select file_name, price, size_list, upsell_product_desc, file_url from catalog_for_website where color_or_style = \'" + option + "\';")\npd_prod_data = table_prod_data.to_pandas() \n\n# assign each column of the row returned to its own variable \nprice = \'$\' + str(pd_prod_data[\'PRICE\'].iloc[0])+\'0\'\nfile_name = pd_prod_data[\'FILE_NAME\'].iloc[0]\nsize_list = pd_prod_data[\'SIZE_LIST\'].iloc[0]\nupsell = pd_prod_data[\'UPSELL_PRODUCT_DESC\'].iloc[0]\nurl = pd_prod_data[\'FILE_URL\'].iloc[0]\n\n# display the info on the page\nst.image(image=url, width=400, caption=product_caption)\nst.markdown(\'**Price:** \'+ price)\nst.markdown(\'**Sizes Available:** \' + size_list)\nst.markdown(\'**Also Consider:** \' + upsell)\n'
session.file.put_stream(io.BytesIO(APP_CODE.encode()), 'snow://streamlit/ZENAS_ATHLEISURE_DB.PRODUCTS."Zena\'s Amazing Athleisure Catalog"/versions/live/streamlit_app.py', auto_compress=False, overwrite=True)
print(APP_CODE)

> ⚠️ **Audit note:** Snowsight assigns the app's object identifier (DLKW_065 shows a generated stage name, UCY5D7...); here the identifier is the title. The stage-less app has no app stage, so the course's 'Load copies of the image files into your app's stage' route is not available; the app's final code (DLKW_073) uses the pre-signed URL instead, which needs no app stage.

## Page: 🥋 Work Out Image Displays (OPTIONAL)

> **Course:** 🥋 Load copies of the images into your app's stage (DLKW_065, 071); 📓 The Second Option: pre-signed URLs (DLKW_072, 073): the app code already uses `url = pd_prod_data['FILE_URL'].iloc[0]` and `st.image(image=url, ...)`.

In [ ]:
%%sql -r app_query_check
-- Not a course cell: the query the app runs for the first colour, checking the pre-signed URL column
select color_or_style, price, size_list, upsell_product_desc, left(file_url, 60) as file_url_start
from catalog_for_website order by color_or_style limit 3;